In [0]:
# Variables de configuración
import os

USERNAME = spark.sql("SELECT current_user()").collect()[0][0]
CATALOG = "main"  # Ajusta según tu catálogo
SCHEMA = "default"  # Ajusta según tu esquema

print(f"USERNAME: {USERNAME}")
print(f"CATALOG: {CATALOG}")
print(f"SCHEMA: {SCHEMA}")

In [0]:
# Instalar kaggle
%pip install kaggle --quiet

# Configurar directorio de trabajo
import os
import zipfile

# Crear directorio para los datos
data_dir = "/tmp/crop_data"
os.makedirs(data_dir, exist_ok=True)
os.chdir(data_dir)

print(f"Directorio de trabajo: {os.getcwd()}")

In [0]:
%undefined
cd /tmp/crop_data

# Nota: Para usar la API de Kaggle, necesitas autenticarte
# Opción 1: Si tienes kaggle.json configurado
# kaggle datasets download -d patelris/crop-yield-prediction-dataset

# Opción 2: Descargar manualmente el archivo
# Por ahora, descargamos usando wget si el dataset es público
wget -q --show-progress https://www.kaggle.com/api/v1/datasets/download/patelris/crop-yield-prediction-dataset -O crop-yield.zip

# Descomprimir
unzip -o crop-yield.zip
ls -lh

In [0]:
# Cargar el CSV en un Spark DataFrame
from pyspark.sql import SparkSession

# Ruta al archivo CSV
csv_path = "/tmp/crop_data/yield_df.csv"

# Leer el CSV con Spark
df = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

print(f"Dataset cargado exitosamente desde: {csv_path}")

In [0]:
# Mostrar el esquema
print("=" * 50)
print("ESQUEMA DEL DATASET")
print("=" * 50)
df.printSchema()

# Conteo total de filas
total_rows = df.count()
print(f"\n{'=' * 50}")
print(f"TOTAL DE FILAS: {total_rows:,}")
print("=" * 50)

In [0]:
# Mostrar 5 filas de ejemplo
print("=" * 50)
print("MUESTRA DE 5 FILAS")
print("=" * 50)
display(df.limit(5))

In [0]:
# Valores únicos en la columna "Item"
print("=" * 50)
print("TIPOS DE CULTIVO (Únicos en 'Item')")
print("=" * 50)

item_values = df.select("Item").distinct().orderBy("Item")
item_count = item_values.count()

print(f"\nTotal de tipos de cultivo: {item_count}\n")
display(item_values)

In [0]:
# Valores únicos en "Area" que contengan Argentina o Brazil
print("=" * 50)
print("AREAS QUE INCLUYEN 'ARGENTINA' O 'BRAZIL'")
print("=" * 50)

from pyspark.sql.functions import col

area_filtered = df.select("Area").distinct() \
    .filter(
        col("Area").contains("Argentina") | 
        col("Area").contains("Brazil")
    ) \
    .orderBy("Area")

filtered_count = area_filtered.count()
print(f"\nTotal de áreas encontradas: {filtered_count}\n")
display(area_filtered)